# Smart Route Optimization in a City Using Multi-Weighted Graphs

## Final Exam Project

**Author:** Zhivko Nedyalkov Georgiev  
**Course:** Math Concepts for Developers

---

## Introduction

In real urban navigation, the best route is not always the shortest by distance.
Drivers and travelers may care about several factors at the same time, such as travel time, traffic congestion, safety, and road quality.

This project models a city as a weighted graph and studies how route optimization changes when each road has multiple attributes instead of a single weight.

The goal is to implement and analyze a shortest path algorithm on a multi-weighted graph and compare the resulting routes under different optimization priorities.


## Problem Statement

A city road network can be modeled as a graph:

- nodes represent intersections or locations
- edges represent roads between them

In the simplest shortest path problem, each road has only one weight, usually distance.
However, this is not realistic in urban environments, where route choice depends on multiple criteria.

In this project, each road segment will have the following attributes:

- distance
- traffic level
- safety risk
- road quality penalty

These attributes will be combined into a single cost function, and Dijkstra's algorithm will be used to find the optimal route.


## Why This Project Matters

Route optimization is one of the most practical applications of graph theory.

Real navigation systems do not optimize only distance, but also factors such as traffic, safety, and road quality.

This project attempts to model a simplified but more realistic version of urban routing.


## Project Objectives

The main objectives of this project are:

1. To model an urban network as a weighted graph.
2. To define a multi-criteria cost function for route selection.
3. To implement Dijkstra's algorithm in Python.
4. To compare different routing scenarios.
5. To analyze the strengths and limitations of the model.


## Mathematical Background

A graph is an ordered pair:

\[
G = (V, E)
\]

where:

- \(V\) is the set of vertices (nodes)
- \(E\) is the set of edges (roads)

A path in the graph is a sequence of connected nodes.
The cost of a path is the sum of the costs of all edges along that path.

In standard shortest path problems, each edge has one weight:

\[
w(e)
\]

and the objective is to minimize:

\[
C(P) = \sum_{e \in P} w(e)
\]

In this project, each edge has several attributes:

\[
d(e) = \text{distance}
\]
\[
t(e) = \text{traffic}
\]
\[
s(e) = \text{safety risk}
\]
\[
q(e) = \text{road quality penalty}
\]

We define a combined cost function:

\[
w(e) = \alpha d(e) + \beta t(e) + \gamma s(e) + \delta q(e)
\]

where:

- \(\alpha\) controls the importance of distance
- \(\beta\) controls the importance of traffic
- \(\gamma\) controls the importance of safety
- \(\delta\) controls the importance of road quality

Thus, the route optimization problem becomes:

\[
\min_{P} \sum_{e \in P} \left(\alpha d(e) + \beta t(e) + \gamma s(e) + \delta q(e)\right)
\]

This allows us to model different routing preferences.

## Why One Weight Is Not Enough

In real life, a route is rarely chosen only by distance.

For example:

- a short road may be heavily congested
- a slightly longer road may be safer
- a road with lower distance may have poor road conditions
 - an emergency vehicle may prefer the fastest route, not the shortest one

Because of this, a more realistic model must include several edge attributes.

This makes the optimization problem more applicable to real urban navigation.


In [ ]:
city_graph = {
    "A": {
        "B": {"distance": 4, "traffic": 3, "safety": 2, "quality": 1},
        "C": {"distance": 2, "traffic": 5, "safety": 3, "quality": 2}
    },
    "B": {
        "D": {"distance": 5, "traffic": 2, "safety": 1, "quality": 2},
        "E": {"distance": 6, "traffic": 4, "safety": 2, "quality": 3}
    },
    "C": {
        "B": {"distance": 2, "traffic": 2, "safety": 2, "quality": 1},
        "D": {"distance": 8, "traffic": 1, "safety": 1, "quality": 1},
        "F": {"distance": 7, "traffic": 5, "safety": 4, "quality": 3}
    },
    "D": {
        "E": {"distance": 2, "traffic": 3, "safety": 1, "quality": 1},
        "G": {"distance": 6, "traffic": 2, "safety": 2, "quality": 2}
    },
    "E": {
        "G": {"distance": 3, "traffic": 4, "safety": 2, "quality": 2}
    },
    "F": {
        "G": {"distance": 2, "traffic": 1, "safety": 3, "quality": 1}
    },
    "G": {}
}
city_graph


## Graph Model

The city is modeled as a directed weighted graph.
Each edge contains four attributes:

- distance
- traffic
- safety
- quality

These attributes are not directly optimized one by one.
Instead, they are combined into a single cost function.

In [ ]:
def edge_cost(edge, alpha=1.0, beta=1.0, gamma=1.0, delta=1.0):
    return (
        alpha * edge["distance"] +
        beta * edge["traffic"] +
        gamma * edge["safety"] +
        delta * edge["quality"]
    )

print(edge_cost(city_graph["A"]["B"], alpha=1, beta=2, gamma=1, delta=1))


## Edge Cost Function

The combined edge cost is computed as a weighted sum of the attributes.

This allows us to simulate different routing preferences:

- distance-focused
- traffic-aware
- safety-aware
- balanced optimization

## Dijkstra Algorithm

We apply Dijkstra's algorithm after converting each edge into a single combined cost.
This works because the resulting weights are non-negative.


In [ ]:
import heapq
import math

def dijkstra_multiweight(graph, start, end, alpha=1.0, beta=1.0, gamma=1.0, delta=1.0):
    distances = {node: math.inf for node in graph}
    previous = {node: None for node in graph}
    distances[start] = 0
    pq = [(0, start)]

    while pq:
        current_distance, current_node = heapq.heappop(pq)

        if current_distance > distances[current_node]:
            continue

        if current_node == end:
            break

        for neighbor, attrs in graph[current_node].items():
            cost = edge_cost(attrs, alpha, beta, gamma, delta)
            new_distance = current_distance + cost

            if new_distance < distances[neighbor]:
                distances[neighbor] = new_distance
                previous[neighbor] = current_node
                heapq.heappush(pq, (new_distance, neighbor))

    path = []
    node = end
    while node is not None:
        path.append(node)
        node = previous[node]
    path.reverse()

    if not path or path[0] != start:
        return None, math.inf

    return path, distances[end]


In [ ]:
def route_details(graph, path):
    details = []
    total_distance = 0
    total_traffic = 0
    total_safety = 0
    total_quality = 0

    for i in range(len(path) - 1):
        edge = graph[path[i]][path[i + 1]]
        details.append((path[i], path[i + 1], edge))
        total_distance += edge["distance"]
        total_traffic += edge["traffic"]
        total_safety += edge["safety"]
        total_quality += edge["quality"]

    return {
        "segments": details,
        "distance": total_distance,
        "traffic": total_traffic,
        "safety": total_safety,
        "quality": total_quality
    }


## Experiment 1: Distance-Only Route


In [ ]:
path_distance, cost_distance = dijkstra_multiweight(city_graph, "A", "G", alpha=1, beta=0, gamma=0, delta=0)
details_distance = route_details(city_graph, path_distance)
print("Distance-only route:", path_distance)
print("Weighted cost:", cost_distance)
print(details_distance)


## Experiment 2: Traffic-Aware Route


In [ ]:
path_traffic, cost_traffic = dijkstra_multiweight(city_graph, "A", "G", alpha=1, beta=2, gamma=0, delta=0)
details_traffic = route_details(city_graph, path_traffic)
print("Traffic-aware route:", path_traffic)
print("Weighted cost:", cost_traffic)
print(details_traffic)
